# Olist E-Commerce: Data Cleaning Pipeline

**Author:** Okunowo Oluwademilade David
**Date:** 2026-07-27

This notebook covers the full cleaning process for the Olist Brazilian
e-commerce dataset (9 raw files: customers, sellers, products, orders,
order items, payments, reviews, geolocation, and category translations).

**What this notebook does:**
- Loads all 9 raw CSVs
- Runs a data quality report (before/after) on each table
- Cleans nulls, duplicates, invalid values, and inconsistent formatting
- Merges everything into one order-level master table
- Exports a clean dataset for use in analysis (see `02_eda_delivery_delay.ipynb`)

**What this notebook does NOT do:**
- Exploratory analysis or answering business questions — that lives in a separate notebook

In [1]:
import pandas as pd
import numpy as np

# 1. LOAD everything

In [2]:
customers = pd.read_csv("olist_customers_dataset.csv", encoding="utf-8-sig")
geolocation = pd.read_csv("olist_geolocation_dataset.csv", encoding="utf-8-sig")
orders_items = pd.read_csv("olist_order_items_dataset.csv", encoding="utf-8-sig")
order_payments = pd.read_csv("olist_order_payments_dataset.csv", encoding="utf-8-sig")
order_reviews = pd.read_csv("olist_order_reviews_dataset.csv", encoding="utf-8-sig")
orders = pd.read_csv("olist_orders_dataset.csv", encoding="utf-8-sig")
products = pd.read_csv("olist_products_dataset.csv", encoding="utf-8-sig")
sellers = pd.read_csv("olist_sellers_dataset.csv", encoding="utf-8-sig")
translation = pd.read_csv("product_category_name_translation.csv", encoding="utf-8-sig")

In [3]:
def quality_report(df, name):
    print(f"\n--- {name} ---")
    print(f"rows: {len(df)}, cols: {df.shape[1]}")
    print(f"duplicate rows: {df.duplicated().sum()}")
    print("nulls per column:")
    nulls = df.isna().sum()
    print(nulls[nulls > 0] if nulls.sum() > 0 else "  none")

# 2. CLEAN: orders

In [4]:
quality_report(orders, "orders RAW")
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")
contradictory = orders[
    (orders.order_status == "delivered") & (orders.order_delivered_customer_date.isna())
]
print(
    f"orders: {len(contradictory)} rows marked delivered with no delivery date (flagged)"
)

quality_report(orders, "orders CLEANED")


--- orders RAW ---
rows: 99441, cols: 8
duplicate rows: 0
nulls per column:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
orders: 8 rows marked delivered with no delivery date (flagged)

--- orders CLEANED ---
rows: 99441, cols: 8
duplicate rows: 0
nulls per column:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


# 3. CLEAN: order_payments

In [5]:
quality_report(order_payments, "order_payments RAW")
payments_agg = order_payments.groupby("order_id", as_index=False).agg(
    total_payment_value=("payment_value", "sum"),
    payment_installments_max=("payment_installments", "max"),
    payment_types=("payment_type", lambda x: ",".join(sorted(set(x)))),
)

quality_report(payments_agg, "order_payments CLEANED (aggregated to order level)")


--- order_payments RAW ---
rows: 103886, cols: 5
duplicate rows: 0
nulls per column:
  none

--- order_payments CLEANED (aggregated to order level) ---
rows: 99440, cols: 4
duplicate rows: 0
nulls per column:
  none


# 4. CLEAN: order_reviews

In [6]:
quality_report(order_reviews, "order_reviews RAW")
order_reviews["review_creation_date"] = pd.to_datetime(
    order_reviews["review_creation_date"], errors="coerce"
)
order_reviews["review_answer_timestamp"] = pd.to_datetime(
    order_reviews["review_answer_timestamp"], errors="coerce"
)
reviews_clean = order_reviews.sort_values("review_creation_date").drop_duplicates(
    subset="order_id", keep="last"
)
quality_report(reviews_clean, "order_reviews CLEANED(latest review per order)")


--- order_reviews RAW ---
rows: 99224, cols: 7
duplicate rows: 0
nulls per column:
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- order_reviews CLEANED(latest review per order) ---
rows: 98673, cols: 7
duplicate rows: 0
nulls per column:
review_comment_title      87121
review_comment_message    57897
dtype: int64


# 5. CLEAN: order_items

In [7]:
quality_report(orders_items, "orders_items RAW")
orders_items["shipping_limit_date"] = pd.to_datetime(
    orders_items["shipping_limit_date"], errors="coerce"
)
orders_items_agg = orders_items.groupby("order_id", as_index=False).agg(
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
    n_unique_sellers=("seller_id", "nunique"),
)
quality_report(orders_items_agg, "orders_items CLEANED (aggregated to order level)")


--- orders_items RAW ---
rows: 112650, cols: 7
duplicate rows: 0
nulls per column:
  none

--- orders_items CLEANED (aggregated to order level) ---
rows: 98666, cols: 5
duplicate rows: 0
nulls per column:
  none


# 6. CLEAN: products

In [8]:
quality_report(products, "products RAW")
products["product_category_name"] = products["product_category_name"].fillna("unknown")
products = products.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
    }
)
products_full = products.merge(translation, on="product_category_name", how="left")
products_full["product_category_name_english"] = products_full[
    "product_category_name_english"
].fillna("unknown")

quality_report(products_full, "products CLEANED (with English category names)")


--- products RAW ---
rows: 32951, cols: 9
duplicate rows: 0
nulls per column:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- products CLEANED (with English category names) ---
rows: 32951, cols: 10
duplicate rows: 0
nulls per column:
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


# 7.CLEAN: geolocation

In [9]:
quality_report(geolocation, "geolocation RAW")
geo = geolocation.drop_duplicates()
in_brazil = geo.geolocation_lat.between(-34, 6) & geo.geolocation_lng.between(-75, -32)
geo_by_zip = (
    geo.groupby("geolocation_zip_code_prefix")
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean"),
        geolocation_city=("geolocation_city", lambda x: x.mode().iloc[0]),
        geolocation_state=("geolocation_state", lambda x: x.mode().iloc[0]),
    )
    .reset_index()
)
quality_report(geo_by_zip, "geolocation CLEANED (one row per zip prefix)")


--- geolocation RAW ---
rows: 1000163, cols: 5
duplicate rows: 261831
nulls per column:
  none

--- geolocation CLEANED (one row per zip prefix) ---
rows: 19015, cols: 5
duplicate rows: 0
nulls per column:
  none


# 8. CLEAN: quick check on customers/sellers 

In [10]:
quality_report(customers, "customers")
quality_report(sellers, "sellers")


--- customers ---
rows: 99441, cols: 5
duplicate rows: 0
nulls per column:
  none

--- sellers ---
rows: 3095, cols: 4
duplicate rows: 0
nulls per column:
  none


# 9. MERGE everything into one master table

In [11]:
master = orders.merge(customers, on="customer_id", how="left")
master = master.merge(payments_agg, on="order_id", how="left")
master = master.merge(
    reviews_clean[["order_id", "review_score", "review_creation_date"]],
    on="order_id",
    how="left",
)
master = master.merge(orders_items_agg, on="order_id", how="left")
master = master.merge(
    geo_by_zip.add_prefix("customer_"),
    left_on="customer_zip_code_prefix",
    right_on="customer_geolocation_zip_code_prefix",
    how="left",
).drop(columns=["customer_geolocation_zip_code_prefix"])

quality_report(master, "FINAL master table")


--- FINAL master table ---
rows: 99441, cols: 25
duplicate rows: 0
nulls per column:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
total_payment_value                 1
payment_installments_max            1
payment_types                       1
review_score                      768
review_creation_date              768
total_price                       775
total_freight                     775
n_items                           775
n_unique_sellers                  775
customer_geolocation_lat          278
customer_geolocation_lng          278
customer_geolocation_city         278
customer_geolocation_state        278
dtype: int64


In [12]:
master.to_csv("olist_master_cleaned.csv", index=False)